# P0 diagnostic — T5 stage 1, no FP16

Mục tiêu: kiểm tra xem lỗi NaN/collapse của P0 có phải do FP16 hay không.

- Cùng dữ liệu stage 1, cùng `target` full, cùng `max_input_length=512`, `max_target_length=512`.
- Chỉ lấy **16.000 train** và **1.000 validation** ngẫu nhiên để vừa quota GPU.
- 1 epoch, `--no-fp16`, batch 4/GPU, grad accumulation 8.
- Lưu checkpoint và eval 1.000 dòng.
- Đây là **diagnostic**, không phải kết quả paper.


In [ ]:
!nvidia-smi
import torch
print("torch:", torch.__version__)
print("cuda devices:", torch.cuda.device_count())


In [ ]:
!rm -rf /kaggle/working/Vuln2Action-Lab
!git clone --branch feature/paper-reproduction https://github.com/Hoang-Minh-Hieu-2806/Vuln2Action-Lab.git /kaggle/working/Vuln2Action-Lab
%cd /kaggle/working/Vuln2Action-Lab
!pip install -q -e ".[train]"


In [ ]:
from pathlib import Path
import glob, os
from datasets import load_dataset

hits = sorted(glob.glob("/kaggle/input/**/flow_train.jsonl", recursive=True))
print("flow_train hits:", hits)
if not hits:
    raise FileNotFoundError("flow_train.jsonl not found under /kaggle/input")
FLOW_DIR = Path(hits[0]).parent
os.environ["FLOW_DIR"] = str(FLOW_DIR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("FLOW_DIR:", FLOW_DIR)

raw = load_dataset(
    "json",
    data_files={
        "train": str(FLOW_DIR / "flow_train.jsonl"),
        "validation": str(FLOW_DIR / "flow_validation.jsonl"),
    },
)
train = raw["train"].shuffle(seed=42).select(range(min(16000, len(raw["train"]))))
val = raw["validation"].shuffle(seed=43).select(range(min(1000, len(raw["validation"]))))

diag_dir = Path("/kaggle/working/diag_no_fp16_dataset")
diag_dir.mkdir(parents=True, exist_ok=True)
train.to_json(str(diag_dir / "flow_train.jsonl"), lines=True)
val.to_json(str(diag_dir / "flow_validation.jsonl"), lines=True)
print("diagnostic train rows:", len(train))
print("diagnostic validation rows:", len(val))
print("train columns:", train.column_names)


In [ ]:
!python -m torch.distributed.run --nproc_per_node=2 scripts/train_t5.py \
  --dataset /kaggle/working/diag_no_fp16_dataset \
  --output-dir /kaggle/working/diag_no_fp16 \
  --model-name t5-base \
  --target-field target \
  --max-input-length 512 \
  --max-target-length 512 \
  --per-device-batch-size 4 \
  --gradient-accumulation-steps 8 \
  --learning-rate 5e-5 \
  --num-train-epochs 1 \
  --save-steps 100 \
  --eval-steps 100 \
  --logging-steps 20 \
  --save-total-limit 2 \
  --no-load-best-model-at-end \
  --no-fp16 \
  --gradient-checkpointing


In [ ]:
!python scripts/evaluate_t5.py \
  --checkpoint /kaggle/working/diag_no_fp16/final \
  --dataset /kaggle/working/diag_no_fp16_dataset \
  --split validation \
  --target-field target \
  --output-dir /kaggle/working/diag_no_fp16_eval1000 \
  --max-input-length 512 \
  --max-target-length 512 \
  --batch-size 4 \
  --num-beams 4 \
  --limit 1000 \
  --no-fp16


In [ ]:
import json
from pathlib import Path

base = Path("/kaggle/working/diag_no_fp16")
print("\n--- training config ---")
cfg = json.loads((base / "training_config.json").read_text(encoding="utf-8"))
print(json.dumps(cfg, indent=2, ensure_ascii=False))

print("\n--- last training log entries ---")
state_path = base / "trainer_state.json"
if state_path.exists():
    state = json.loads(state_path.read_text(encoding="utf-8"))
    rows = [x for x in state.get("log_history", []) if "loss" in x or "eval_loss" in x]
    for row in rows[-20:]:
        print(json.dumps(row, ensure_ascii=False))
else:
    print("missing", state_path)

out = Path("/kaggle/working/diag_no_fp16_eval1000")
m = json.loads((out / "metrics.json").read_text(encoding="utf-8"))
print("\n--- metrics ---")
print(json.dumps(m, indent=2, ensure_ascii=False))

pred_path = out / "predictions.jsonl"
rows = [json.loads(line) for line in pred_path.read_text(encoding="utf-8").splitlines()]
empty = sum(1 for r in rows if not r.get("prediction", "").strip())
print(f"\nempty predictions: {empty}/{len(rows)}")
print("\n--- samples ---")
for i, r in enumerate(rows[:3]):
    print(f"\n[{i}] {r.get('cve_id')}")
    print("REF:", (r.get("reference") or "")[:300])
    print("PRED:", (r.get("prediction") or "")[:300])


## Cách đọc kết quả

- Nếu `eval_loss` và `grad_norm` vẫn hữu hạn, prediction không rỗng: FP16 là nghi phạm chính; cần chạy lại full data bằng FP32 sau khi quota reset.
- Nếu FP32 vẫn NaN/rỗng: lỗi không chỉ do FP16; phải kiểm tra tiếp target length, dữ liệu, hoặc chuyển sang `target_steps` với ghi chú deviation.
- Kết quả này chỉ trên 16.000 train/1.000 validation, không dùng làm số paper.
